# DuckDB Analytics + Elasticsearch Search

**Routing pattern:**
- WRITE: `[duckdb]` — write to SQLite/Parquet via DuckDB driver
- READ: `[duckdb]` — browse/paginate through Parquet files
- SEARCH: `[elasticsearch]` — spatial/attribute filtered queries via ES
- METADATA: `[postgresql]` — collection metadata in PG

Optimal for analytical datasets already in Parquet format that need ES-powered search.

In [ ]:
import httpx

BASE = "http://localhost:8000"
CATALOG_ID = "demo-analytics"
COLLECTION_ID = "crop-yield-2024"

client = httpx.AsyncClient(base_url=BASE, timeout=30)

def _check(r, label=""):
    status = r.status_code
    ok = status < 400
    suffix = "" if ok else f" — {r.text[:300]}"
    print(f"{label + ': ' if label else ''}{status}{suffix}")
    return r

print("Client ready")

In [ ]:
r = await client.post("/stac/catalogs", json={"id": CATALOG_ID, "title": "Analytics Demo", "description": "Analytics Demo catalog."})
_check(r)

In [ ]:
r = await client.post(f"/stac/catalogs/{CATALOG_ID}/collections", json={
    "id": COLLECTION_ID, "title": "Crop Yield 2024",
    "description": "Agricultural yield data stored as Parquet, searchable via ES",
    "license": "CC-BY-4.0",
    "extent": {"spatial": {"bbox": [[-180, -90, 180, 90]]}, "temporal": {"interval": [["2020-01-01T00:00:00Z", None]]}},
})
_check(r)

In [ ]:
routing = {
    "operations": {
        "WRITE": [
            {"driver_id": "CollectionDuckdbDriver", "on_failure": "fatal", "write_mode": "sync"},
            {"driver_id": "CollectionElasticsearchDriver", "on_failure": "warn", "write_mode": "async"}
        ],
        "READ": [{"driver_id": "CollectionDuckdbDriver"}],
        "SEARCH": [{"driver_id": "CollectionElasticsearchDriver"}],
        "METADATA": [{"driver_id": "CollectionPostgresqlDriver"}]
    }
}
r = await client.put(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/configs/CollectionRoutingConfig",
    json=routing,
)
_check(r)

In [ ]:
# Write policy: UPDATE in place (overwrite by external_id if re-ingested)
policy = {
    "plugin_id": "collection:write_policy",
    "on_conflict": "update",
    "track_asset_id": True,
    "enable_validity": False,
    "external_id_field": "id"
}
r = await client.put(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/configs/collection:write_policy",
    json=policy,
)
_check(r)

In [ ]:
import random

features = [
    {
        "type": "Feature",
        "id": f"field-{i:04d}",
        "geometry": {"type": "Point", "coordinates": [10.0 + random.uniform(0, 5), 40.0 + random.uniform(0, 5)]},
        "properties": {
            "crop": random.choice(["wheat", "corn", "rice", "soybean"]),
            "yield_t_ha": round(random.uniform(2.0, 8.0), 2),
            "country": random.choice(["IT", "ES", "FR", "DE"]),
        }
    }
    for i in range(20)
]
r = await client.post(
    f"/stac/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items",
    json={"type": "FeatureCollection", "features": features},
)
_check(r, "20 crop features written to DuckDB")

In [ ]:
# READ operation → DuckDB (streaming from Parquet/SQLite)
r = await client.get(f"/stac/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items?limit=5")
data = r.json()
print(f"READ from DuckDB: {len(data.get('features', []))} items")
for f in data.get("features", []):
    p = f["properties"]
    print(f"  {f['id']}: {p.get('crop')} yield={p.get('yield_t_ha')} t/ha")

In [ ]:
# SEARCH operation → Elasticsearch (bbox + attribute filter)
r = await client.post(
    f"/search/catalogs/{CATALOG_ID}/items-search",
    json={
        "collections": [COLLECTION_ID],
        "bbox": [10.0, 40.0, 15.0, 45.0],
        "filter": {"op": "=", "args": [{"property": "crop"}, "wheat"]},
        "limit": 10,
    },
)
data = r.json()
print(f"SEARCH from ES (wheat in bbox): {len(data.get('features', []))} results")

In [ ]:
# DuckDB-specific: export to Parquet
r = await client.post(
    f"/storage/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/export",
    json={"format": "parquet", "target_path": "/tmp/crop_yield_2024.parquet"},
)
print(r.status_code, r.json() if r.status_code == 200 else r.text[:200])